# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [11]:
# !pip install langchain_community langchain_experimental langchain_openai
# %pip install langchain-experimental
# %pip install faiss-cpu
# %pip install google-search-results
%pip install langchain_google_genai

Note: you may need to restart the kernel to use updated packages.


In [5]:
from typing import Optional

# from langchain.chains import LLMChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_experimental.autonomous_agents import BabyAGI
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [6]:
# %pip install faiss-cpu > /dev/null
# %pip install google-search-results > /dev/null
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [16]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

# OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
SERPAPI_API_KEY = os.getenv('SERP_API_KEY')
GEMINI_API_KEY= os.getenv('GEMINI_API_KEY')

In [38]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Define your embedding model
# embeddings_model = OpenAIEmbeddings()
embeddings_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
# Initialize the vectorstore as empty
import faiss

embedding_size = 3072
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [61]:
# from langchain.agents import AgentExecutor
# from langchain.chains import LLMChain
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import OpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=GEMINI_API_KEY,
    temperature=0.0)


todo_prompt = ChatPromptTemplate.from_template(
    "You are a planner who is an expert at coming up with a todo list for a given objective. Come up with a todo list for this objective: {objective}"
)

todo_chain = todo_prompt | llm

search = SerpAPIWrapper(
    serpapi_api_key=SERPAPI_API_KEY
)

# tools = [
#     Tool(
#         name="Search",
#         func=search.run,
#         description="useful for when you need to answer questions about current events",
#     ),
#     Tool(
#         name="TODO",
#         func=todo_chain.run,
#         description="useful for when you need to come up with todo lists. Input: an objective to create a todo list for. Output: a todo list for that objective. Please be very clear what the objective is!",
#     ),
# ]

@tool
def search(query: str) -> str:
    """Useful for when you need to answer questions about current events or real-time information."""
    return search.run(query)

@tool
def todo(objective: str) -> str:
    """Useful for when you need to come up with todo lists. 
    Input should be an objective to create a todo list for.
    """
    # Invoke the LCEL chain and return the string content
    response = todo_chain.invoke({"objective": objective})
    return response.content

tools = [search, todo]


# prefix = """You are an AI who performs one task based on the following objective: {objective}. Take into account these previously completed tasks: {context}."""
# suffix = """Question: {task}
# {agent_scratchpad}"""
# prompt = ZeroShotAgent.create_prompt(
#     tools,
#     prefix=prefix,
#     suffix=suffix,
#     input_variables=["objective", "task", "context", "agent_scratchpad"],
# )

template = """You are an AI who performs one task based on the following objective: {objective}. 
Take into account these previously completed tasks: {context}.

You have access to the following tools:
{tools}

Use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Question: {task}
Thought: {agent_scratchpad}"""

prompt = ChatPromptTemplate.from_template(template)

In [62]:
# llm = OpenAI(temperature=0)
# llm_chain = LLMChain(llm=llm, prompt=prompt)
# tool_names = [tool.name for tool in tools]
# agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
# agent_executor = AgentExecutor.from_agent_and_tools(
#     agent=agent, tools=tools, verbose=True
# )


from langchain.agents import create_agent
from langchain_classic.agents import create_react_agent 
from langchain_classic.agents import AgentExecutor

# llm = OpenAI(temperature=0)
llm_chain = prompt | llm
# tool_names = [tool.name for tool in tools]

agent = create_react_agent(
    llm, 
    tools=tools, 
    prompt=prompt
)
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True,
    handle_parsing_errors=True # Recommended to catch text-formatting hiccups
)

### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [51]:
OBJECTIVE = "Write a weather report for SF today"

In [58]:
# Logging of LLMChains
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    verbose=True,
    max_iterations=3,
)


In [64]:
baby_agi.invoke({"objective": OBJECTIVE})
# baby_agi({"objective": OBJECTIVE})


*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new TaskExecutionChain chain...
Prompt after formatting:
You are an AI who performs one task based on the following objective: Write a weather report for SF today.Take into account these previously completed tasks: . Your task: Make a todo list. Response:

> Finished chain.

*****TASK RESULT*****

Here is your todo list:

*   Check current weather conditions for San Francisco.
*   Gather temperature data (high/low).
*   Note precipitation chances/amounts.
*   Identify wind speed and direction.
*   Look for any special weather advisories (fog, sun, etc.).
*   Draft the weather report.
*   Review for clarity and accuracy.


GoogleGenerativeAIError: Error embedding content: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}